# Radar water through the seasons, middle Mahakam (Python)

**Fullstack Remote Sensing** by Rifky Nauval Hendrawan · [Chapter: Synthetic Aperture Radar Fusion](https://rifkynhsp-byte.github.io/Fullstack-Remote-Sensing/en/10-sar-fusion.html) · [Bab (Bahasa Indonesia)](https://rifkynhsp-byte.github.io/Fullstack-Remote-Sensing/id/10-sar-fusion.html) · [Open the JavaScript in the Code Editor](https://code.earthengine.google.com/?scriptPath=users/rifkynauvalhsp/Fullstack-Remote-Sensing:ch10_sar_water_mahakam)

Run the cells from top to bottom. The first cell installs what Colab lacks, fetches the book's data and helpers, and signs you in to Earth Engine with your own Google Cloud project ([how to get one](https://developers.google.com/earth-engine/guides/access)).

*Jalankan sel dari atas ke bawah. Sel pertama memasang paket, mengambil data buku, dan masuk ke Earth Engine dengan proyek Google Cloud Anda sendiri.*

In [ ]:
# @title Setup: packages, book data, Earth Engine sign-in
!pip -q install earthengine-api geemap tabulate
!test -d /content/book || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/rifkynhsp-byte/Fullstack-Remote-Sensing.git /content/book
!git -C /content/book sparse-checkout set tools scripts data   # only what the code needs, not the website
import sys, os; sys.path[:0] = ['/content/book/tools', '/content/book/scripts/py']; os.chdir('/content/book')
__file__ = '/content/book/scripts/py/ch10_sar_water_mahakam.py'   # the script finds the book's data relative to itself
PROJECT = "your-cloud-project"  # @param {type:"string"}
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)
_init = ee.Initialize; ee.Initialize = lambda *a, **k: None   # the script must not sign in again

CHAPTER 10 | Sentinel-1 water frequency and monthly flooded area over the
middle Mahakam floodplain, 2017 to 2024 (GEE101 study area).

In [ ]:
import ee
import matplotlib.pyplot as plt
import pandas as pd

roi = ee.Geometry.Polygon([[[115.988, -0.083], [117.020, -0.203],
                            [117.285, 0.919], [116.269, 1.114]]])

s1 = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(roi)
      .filterDate("2017-01-01", "2025-01-01")
      .filter(ee.Filter.eq("instrumentMode", "IW"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
      .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
      .select("VV"))

THRESHOLD = -13
water = s1.map(lambda img: img.focal_median(100, "circle", "meters").lt(THRESHOLD)
               .rename("water").copyProperties(img, ["system:time_start"]))
frequency = water.mean().clip(roi)

In [ ]:
def month_feature(i):
    start = ee.Date("2017-01-01").advance(i, "month")
    m = water.filterDate(start, start.advance(1, "month"))
    km2 = (m.max().multiply(ee.Image.pixelArea()).divide(1e6)
           .reduceRegion(reducer=ee.Reducer.sum(), geometry=roi, scale=100,
                         maxPixels=1e10, tileScale=4).get("water"))
    return ee.Feature(None, {"month": start.format("YYYY-MM"), "water_km2": km2,
                             "passes": m.size()})


monthly = (ee.FeatureCollection(ee.List.sequence(0, 95).map(month_feature))
           .filter(ee.Filter.gt("passes", 0)))

In [ ]:
def monthly_frame():
    """Ask for one year at a time.

    Asking for all 96 monthly sums in one request fails with "Too many
    concurrent aggregations". Twelve at a time succeeds. This batching is
    trivial in Python and awkward in the Code Editor.
    """
    frames = []
    for year in range(8):
        batch = (ee.FeatureCollection(ee.List.sequence(12 * year, 12 * year + 11)
                                      .map(month_feature))
                 .filter(ee.Filter.gt("passes", 0)))
        frames.append(pd.DataFrame([f["properties"] for f in batch.getInfo()["features"]]))
    return pd.concat(frames, ignore_index=True)

In [ ]:
def plot_monthly_water(df):
    """Monthly maximum water extent. Find the wet seasons, and the dry years."""
    df = df.assign(month=pd.to_datetime(df["month"])).sort_values("month")
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.plot(df["month"], df["water_km2"], color="#2171b5", lw=1)
    ax.fill_between(df["month"], df["water_km2"], df["water_km2"].min(),
                    color="#2171b5", alpha=0.15)
    ax.set_ylabel("Water extent (km²)")
    ax.set_title("Middle Mahakam floodplain, Sentinel-1 descending, VV < −13 dB",
                 loc="left", fontsize=9)
    return fig

In [ ]:
def plot_month_of_year(df):
    """The average year: which months flood?"""
    df = df.assign(m=pd.to_datetime(df["month"]).dt.month)
    g = df.groupby("m")["water_km2"]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(g.median().index, g.median().values, color="#6baed6")
    ax.errorbar(g.median().index, g.median().values,
                yerr=[g.median() - g.quantile(0.1), g.quantile(0.9) - g.median()],
                fmt="none", color="#08306b", lw=1)
    ax.set_xticks(range(1, 13))
    ax.set_xlabel("Month")
    ax.set_ylabel("Water extent (km²)")
    ax.set_title("Median year, with 10th to 90th percentile", loc="left", fontsize=9)
    return fig

In [ ]:
def products():
    return [
        {"kind": "map", "name": "ch10-water-frequency", "image": frequency, "region": roi,
         "vis": {"min": 0, "max": 1,
                 "palette": ["ffffff", "c6dbef", "6baed6", "2171b5", "08306b"]},
         "legend": "Share of Sentinel-1 passes classed as water, 2017 to 2024",
         "title": "How often is it water?",
         "source": "Sentinel-1 GRD (Copernicus), descending, VV. GEE.",
         "caption": "Water frequency across 2017 to 2024. Dark blue is permanent water; "
                    "the pale fringes are the floodplain that comes and goes."},
        {"kind": "chart", "name": "ch10-water-monthly", "data": monthly_frame,
         "plot": plot_monthly_water,
         "caption": "Monthly maximum water extent, 2017 to 2024. Look for the wet "
                    "seasons and the dry years; optical sensors would see most of these "
                    "months as cloud."},
        {"kind": "chart", "name": "ch10-water-season", "data": monthly_frame,
         "plot": plot_month_of_year,
         "caption": "The same data folded into one year: the flood season and how much "
                    "it varies between years."},
    ]

### The chapter's maps, charts and tables
The same products the book shows, drawn here. Maps also open as an interactive layer.

In [ ]:
from colab_render import show_all
show_all(products())